# Llama 3.1 8B Fine-Tuning for ERP Finance Module
This notebook uses [Unsloth](https://github.com/unslothai/unsloth) to perform QLoRA fine-tuning. Unsloth provides 2x faster training and 70% less VRAM usage, making it ideal for the free Google Colab T4 GPU.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# 1. Upload your train.jsonl file to the Colab environment
# 2. Load the dataset
dataset = load_dataset("json", data_files="train.jsonl", split="train")

def format_prompts(examples):
    conversations = []
    for instruction, input_str, output in zip(examples["instruction"], examples["input"], examples["output"]):
        # For MCP tool-calling and context adherence, we inject a strict system prompt.
        system_prompt = "You are an expert ERP Finance AI assistant. You process financial events accurately and do not hallucinate outside the provided context."
        
        user_content = instruction
        if input_str.strip():
            user_content += f"\nContext: {input_str}"
            
        convo = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        conversations.append(convo)
        
    # Apply the tokenizer's chat template
    formatted_texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in conversations]
    return { "text" : formatted_texts }

dataset = dataset.map(format_prompts, batched = True)
print("Sample Formatted Text:")
print(dataset[0]['text'])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set to 1 for a full pass over 25k records
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

### Export to GGUF for Ollama
Now we export the fine-tuned model into GGUF format so you can run it locally with Ollama.

In [ ]:
# Save the model weights (LoRA adapters)
model.save_pretrained("erp_finance_lora") 
tokenizer.save_pretrained("erp_finance_lora")

# Export to GGUF format
# 'q4_k_m' is a good balance of size and quality (4-bit quantization)
model.save_pretrained_gguf(
    "erp_finance_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m"
)

print("GGUF Export complete. Download the .gguf file from the 'erp_finance_gguf' folder and transfer it to your local machine.")